# Tutorial 8 — Molecular Generation: Optimizing a Docking Score

**Docking tutorial we are cheaply adapting:**

> Volkamer Lab / TeachOpenCADD, *T015 · Protein-Ligand Docking*.
> https://projects.volkamerlab.org/teachopencadd/talktorials/T015_protein_ligand_docking.html

**Core idea.** Protein-ligand docking (here: `smina`, a fork of AutoDock Vina) scores how well a small
molecule fits into a protein binding pocket — a cheap-but-noisy proxy for binding affinity. A real
drug-discovery campaign doesn't just dock one molecule, it *searches* chemical space for molecules
that dock well. This notebook builds and compares four ways to do that search, all against the same
target: **EGFR** (PDB `2ITO`), the same protein T015 uses.

**What you will do in this notebook, gradually:**

- **(a)** Set up a cheap docking oracle: fetch the protein structure, define the binding pocket, and
  wrap `smina` in a Python function you can call on any SMILES string — then visualize a docked pose.
- **(b)** **Method 1 — random search.** Generate random molecules (via SELFIES, which can't produce
  an invalid string) and see how good the best one gets, purely by luck. A deliberately poor baseline.
- **(c)** **Method 2 — a genetic algorithm.** Evolve a population of molecules with selection,
  crossover, and mutation — a classic, still widely-used approach in molecular design.
- **(d)** **Method 3 — surrogate-guided active learning.** Train a cheap machine-learning model to
  *predict* the docking score, and only spend real docking calls on the candidates it's most excited
  about — a simplified Bayesian-optimization loop, directly useful when the real oracle (docking, or
  a wet-lab assay) is too expensive to call on everything.
- **(e)** **Method 4 — VAE latent-space active learning.** Same idea as Method 3, but candidates come
  from perturbing a pretrained variational autoencoder's continuous latent space instead of mutating
  SELFIES tokens directly.
- **Challenge:** submit your single best candidate molecule to the live class leaderboard, which is
  graded independently (and more accurately) than your own exploratory runs.
- **Bonus:** what changes if you care about drug-likeness and toxicity risk too, not just the docking
  score?

**A note on scale.** Real docking studies use `exhaustiveness=8, num_modes=10` (smina's/T015's own
demo defaults) and screen thousands to millions of molecules. That does not fit in a 30-minute budget
of a live class. This notebook uses much cheaper docking settings (`exhaustiveness=1, num_modes=1`)
and small oracle-call budgets throughout so every result is genuine (real docking, not synthetic),
while staying fast enough to run cell-by-cell in class and re-run multiple times. If docking still
proves too slow or unavailable in your session, a one-line switch (`ORACLE_MODE`) falls back to a
cheap RDKit-descriptor-based proxy objective instead — see part (a).

## 0 · Setup

Run this once. This notebook deliberately needs **no PyTorch/GPU** — the only machine-learning model
used (part (d)) is a small `RandomForestRegressor`, so installs and reruns stay fast. The one binary
dependency, `smina` (docking) plus `openbabel` (file conversion), is a conda-forge-only package —
Colab has no conda by default, so we install it via `condacolab`, the same pattern already used for
`xtb` in the `molecular-representations` tutorial. `tqdm` progress bars show up on every cell that
makes real (slow) oracle calls, so you can see roughly how long a run will take partway through it.


In [ ]:
!pip -q install rdkit selfies scikit-learn ipywidgets tqdm > /dev/null

import io, os, re, json, random, subprocess, time
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator, FilterCatalog, Draw

import selfies as sf
from sklearn.ensemble import RandomForestRegressor
from tqdm.auto import tqdm

RDLogger.DisableLog("rdApp.*")  # RDKit is noisy about invalid SMILES -- we generate plenty of those on purpose

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("rdkit", Chem.rdBase.rdkitVersion, "| selfies", sf.__version__)


In [ ]:
# smina + openbabel (needed for part (a)) are conda-forge-only binaries -- Colab has no conda by
# default. Uncomment and run this cell ONLY on Colab; it installs conda via condacolab and RESTARTS
# the Python runtime, so re-run the notebook from the top afterward (don't continue cell-by-cell in
# the same runtime session -- Runtime > Run all is the easiest way). Skip this cell entirely if
# running locally with a conda env that already has smina/openbabel installed.
# If you uncomment this cell make sure to remove indentation (no spacing before the ! or start of
# the line), otherwise it will not run properly.

# !pip install -q condacolab
# import condacolab
# condacolab.install()


In [ ]:
# Run this cell only after condacolab.install() above has restarted the runtime (Colab only).
# !mamba install -y -c conda-forge smina openbabel

from openbabel import pybel
print("openbabel/pybel import OK")


## (a) · The docking oracle (EGFR, PDB 2ITO)

We reuse T015's exact target and pocket definition, but skip its heavier dependencies
(`opencadd`/MDAnalysis/`teachopencadd`) in favor of plain text-parsing of the raw PDB file — the
`smina` docking itself is unchanged, just cheaper settings and a smaller, self-contained toolchain.

**Target**: EGFR kinase domain, PDB `2ITO`, co-crystallized with an approved kinase-inhibitor ligand
(residue name `IRE`). We deliberately never print that ligand's actual structure or SMILES anywhere in
this notebook — it happens to be an excellent binder for this exact pocket, so revealing it would hand
you a shortcut answer for the challenge below instead of a result you found yourself. The binding
pocket is defined the same way T015 defines it: a box centered on the crystallized ligand, sized to
the ligand's extent plus a 5 Å buffer in every direction — purely from its 3D *coordinates* in the PDB
file, never its chemical structure.


In [ ]:
PDB_ID = "2ITO"
LIGAND_RESNAME = "IRE"

PDB_URL = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
with urllib.request.urlopen(PDB_URL, timeout=30) as r:
    pdb_text = r.read().decode("utf-8", errors="ignore")

print(f"Downloaded {len(pdb_text.splitlines())} lines for {PDB_ID}.")
print(pdb_text[:300])


In [ ]:
# SOLUTION: keep "ATOM" lines (the protein) and pull ligand coordinates from "HETATM" lines
# matching LIGAND_RESNAME, using T015's own pocket-center/size formula.
protein_lines = []
ligand_coords = []
for line in pdb_text.splitlines():
    record = line[:6].strip()
    if record == "ATOM":
        protein_lines.append(line)
    elif record == "HETATM":
        resname = line[17:20].strip()
        if resname == LIGAND_RESNAME:
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            ligand_coords.append([x, y, z])

ligand_coords = np.array(ligand_coords)
pocket_center = (ligand_coords.max(axis=0) + ligand_coords.min(axis=0)) / 2
pocket_size = (ligand_coords.max(axis=0) - ligand_coords.min(axis=0)) + 5

with open("protein.pdb", "w") as f:
    f.write("\n".join(protein_lines) + "\nEND\n")

print(f"{len(protein_lines)} protein atom lines, {len(ligand_coords)} ligand atoms")
print("pocket_center (Angstrom):", pocket_center)
print("pocket_size   (Angstrom):", pocket_size)


### How does `smina` actually score a pose?

**[smina](https://sourceforge.net/projects/smina/)** ("**S**coring and **Min**imization with
AutoDock Vin**a**") is the docking engine we wrap below. Its scoring function is a physics-based
*empirical* sum of pairwise atom-type interaction terms — steric/van der Waals repulsion, hydrophobic
contact, hydrogen bonding, an electrostatic-like term, and a penalty tied to the ligand's flexibility
(rotatable bonds) — with weights fit against experimental binding data rather than derived from first
principles. To find the best pose, it runs a stochastic global conformational search (iterated local
search: random moves + local energy minimization) confined to the docking box you define, and reports
the lowest-scoring pose(s) it finds — the same box/settings that also make it fast enough to run live,
many times, in this class. See Koes, Baumgartner &amp; Camacho, *J. Chem. Inf. Model.* **2013**, *53*,
1893–1904 (`https://doi.org/10.1021/ci300604z`) for the details.


In [ ]:
def pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4):
    """Protein PDB -> PDBQT: add hydrogens at physiological pH, assign partial charges."""
    molecule = next(pybel.readfile("pdb", str(pdb_path)))
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)


def smiles_to_pdbqt(smiles, pdbqt_path, pH=7.4):
    """Ligand SMILES -> 3D PDBQT: embed a 3D conformer (MMFF94s) before writing."""
    molecule = pybel.readstring("smi", smiles)
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    molecule.make3D(forcefield="mmff94s", steps=10000)
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)


def canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return None if mol is None else Chem.MolToSmiles(mol)


def run_smina(ligand_path, protein_path, out_path, pocket_center, pocket_size,
              num_modes=1, exhaustiveness=1, seed=SEED):
    result = subprocess.run(
        ["smina",
         "--ligand", str(ligand_path), "--receptor", str(protein_path), "--out", str(out_path),
         "--center_x", str(pocket_center[0]), "--center_y", str(pocket_center[1]),
         "--center_z", str(pocket_center[2]),
         "--size_x", str(pocket_size[0]), "--size_y", str(pocket_size[1]),
         "--size_z", str(pocket_size[2]),
         "--num_modes", str(num_modes), "--exhaustiveness", str(exhaustiveness), "--seed", str(seed)],
        capture_output=True, text=True, timeout=120,
    )
    return result.stdout


def parse_best_affinity(smina_stdout):
    """Best (mode 1) affinity in kcal/mol from smina's printed results table, or None."""
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", smina_stdout, re.MULTILINE)
    return float(match.group(1)) if match else None


pdb_to_pdbqt("protein.pdb", "protein.pdbqt")
print("protein.pdbqt written")


In [ ]:
# Benchmark: how long does one docking call actually take on this runtime, at cheap vs. T015's
# own demo settings? Run this once before committing to the oracle-call budgets below.
#
# DEMO_SMILES is caffeine -- a completely ordinary, publicly-known molecule with no special claim on
# this pocket. We use it here (and everywhere else a concrete example molecule is needed) instead of
# the real co-crystallized ligand, precisely so nothing in this notebook hands you a shortcut answer.
DEMO_SMILES = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"  # caffeine

smiles_to_pdbqt(DEMO_SMILES, "ligand_demo.pdbqt")

ORACLE_TIMING = {}  # exhaustiveness -> wall-clock seconds for one docking call, reused later to cost
                     # the active-learning surrogate against the real oracle without extra docking calls
for exh, n_modes in [(1, 1), (8, 10)]:
    t0 = time.time()
    stdout = run_smina("ligand_demo.pdbqt", "protein.pdbqt", f"poses_exh{exh}.sdf",
                        pocket_center, pocket_size, num_modes=n_modes, exhaustiveness=exh)
    dt = time.time() - t0
    ORACLE_TIMING[exh] = dt
    affinity = parse_best_affinity(stdout)
    print(f"exhaustiveness={exh:2d}  num_modes={n_modes:2d}  time={dt:5.1f}s  "
          f"best affinity={affinity} kcal/mol")

print("\nThis is a timing benchmark, not a claim about caffeine's true binding affinity -- caffeine")
print("isn't expected to be a strong EGFR binder, it's just a convenient, uninteresting test molecule.")

In [ ]:
# Visualize what the oracle actually does: render the docked pose from the benchmark above (the
# protein pocket plus caffeine's best-scoring conformation) directly in the notebook.
!pip -q install py3Dmol > /dev/null
import py3Dmol


def show_docked_pose(receptor_pdb_path, ligand_poses_sdf_path):
    with open(receptor_pdb_path) as f:
        receptor_pdb_text = f.read()
    with open(ligand_poses_sdf_path) as f:
        ligand_sdf_text = f.read()

    view = py3Dmol.view(width=600, height=420)
    receptor_model = view.addModel(receptor_pdb_text, "pdb")
    receptor_model.setStyle({}, {"cartoon": {"color": "spectrum", "opacity": 0.6}})
    ligand_model = view.addModel(ligand_sdf_text, "sdf")
    ligand_model.setStyle({}, {"stick": {"colorscheme": "greenCarbon"}})
    view.zoomTo({"model": 1})
    view.zoom(0.6)
    return view


show_docked_pose("protein.pdb", "poses_exh8.sdf").show()


In [ ]:
EXHAUSTIVENESS_CHEAP = 1
NUM_MODES_CHEAP = 1
ORACLE_CACHE = {}

# SOLUTION: canonicalize + cache-check first (never dock twice), then dock via smiles_to_pdbqt +
# run_smina + parse_best_affinity, with any failure caught and cached as None.
def dock_smiles(smiles, exhaustiveness=EXHAUSTIVENESS_CHEAP, num_modes=NUM_MODES_CHEAP):
    canon = canonical_smiles(smiles)
    if canon is None:
        return None
    cache_key = (canon, exhaustiveness, num_modes)
    if cache_key in ORACLE_CACHE:
        return ORACLE_CACHE[cache_key]
    try:
        smiles_to_pdbqt(canon, "candidate.pdbqt")
        stdout = run_smina("candidate.pdbqt", "protein.pdbqt", "candidate_poses.sdf",
                            pocket_center, pocket_size, num_modes=num_modes, exhaustiveness=exhaustiveness)
        score = parse_best_affinity(stdout)
    except Exception:
        score = None
    ORACLE_CACHE[cache_key] = score
    return score

print("dock_smiles(caffeine) =", dock_smiles(DEMO_SMILES), "kcal/mol")


In [ ]:
def descriptor_fallback_score(smiles):
    """Cheap RDKit-only proxy objective, used if ORACLE_MODE == "descriptor_fallback": combines
    QED, closeness of logP to 2.5, and H-bond donor count into one 'lower is better' score, matching
    docking affinity's sign convention so every method below can stay oracle-agnostic."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    qed = Descriptors.qed(mol)
    logp = Descriptors.MolLogP(mol)
    n_hdonors = rdMolDescriptors.CalcNumHBD(mol)
    return -10.0 * qed + abs(logp - 2.5) + 0.5 * n_hdonors


ORACLE_MODE = "docking"  # switch to "descriptor_fallback" if docking is too slow/unavailable in your session


def evaluate(smiles, exhaustiveness=EXHAUSTIVENESS_CHEAP, num_modes=NUM_MODES_CHEAP):
    """The one oracle every method below calls -- never dock_smiles/descriptor_fallback_score directly."""
    if ORACLE_MODE == "docking":
        return dock_smiles(smiles, exhaustiveness=exhaustiveness, num_modes=num_modes)
    elif ORACLE_MODE == "descriptor_fallback":
        return descriptor_fallback_score(smiles)
    raise ValueError(f"Unknown ORACLE_MODE: {ORACLE_MODE}")


print("evaluate(caffeine) =", evaluate(DEMO_SMILES))


## (b) · Method 1 — random search baseline

The simplest possible strategy: generate random molecules and keep the best one. This is deliberately
**a poor baseline** — both other methods below use the oracle's feedback to bias where they look next,
so they should have no trouble beating it. It's here so "better than random" is a real, measured bar
to clear, not an assumption.

We generate candidates as random **SELFIES** strings rather than random SMILES, because SELFIES is
designed so that *every* string of valid tokens decodes to a chemically valid molecule (unlike SMILES,
where random character strings are almost always invalid) — so we don't waste oracle calls on garbage.

We build a small token vocabulary from a handful of real, well-known small molecules (common drugs
and a couple of tiny aromatic fragments), rather than using SELFIES's full symbol space, so random
molecules stay reasonably small and drug-like.


In [ ]:
SEED_SMILES = [
    DEMO_SMILES,                       # caffeine
    "CC(=O)Oc1ccccc1C(=O)O",           # aspirin
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",       # ibuprofen
    "CC(=O)Nc1ccc(O)cc1",               # paracetamol
    "CN1CCCC1c1cccnc1",                 # nicotine
    "CCOC(=O)c1ccc(N)cc1",              # benzocaine
    "Fc1ccccc1",                        # fluorobenzene -- keeps F in the vocabulary
    "Clc1ccccc1",                       # chlorobenzene -- keeps Cl in the vocabulary
]


def build_vocab(smiles_list):
    alphabet = set()
    for smi in smiles_list:
        try:
            alphabet |= sf.get_alphabet_from_selfies([sf.encoder(smi)])
        except Exception:
            continue
    return sorted(alphabet)


VOCAB = build_vocab(SEED_SMILES)
print(f"{len(VOCAB)} SELFIES tokens in vocabulary:")
print(VOCAB)


In [ ]:
MAX_HEAVY_ATOMS = 30           # keep generated molecules small -- both drug-like and fast to dock
SELFIES_MIN_LEN, SELFIES_MAX_LEN = 5, 25

# These three things ARE the search space: every method in this notebook (random search, the GA, and
# active learning) draws exclusively from strings built out of VOCAB, between SELFIES_MIN_LEN and
# SELFIES_MAX_LEN tokens long, filtered down to <= MAX_HEAVY_ATOMS heavy atoms. Only the SAMPLING
# STRATEGY differs between methods below -- the space itself never changes, which is what makes the
# sample-efficiency comparison later on a fair one.
n_raw_strings = sum(len(VOCAB) ** length for length in range(SELFIES_MIN_LEN, SELFIES_MAX_LEN + 1))
print(f"vocabulary size       : {len(VOCAB)} tokens")
print(f"string length range   : {SELFIES_MIN_LEN}-{SELFIES_MAX_LEN} tokens")
print(f"heavy-atom cap        : {MAX_HEAVY_ATOMS}")
print(f"raw token combinations: ~{n_raw_strings:.2e}  (before dropping invalid/oversized ones -- "
      f"still an astronomically bigger space than any budget below can exhaustively search)")


In [ ]:
# SOLUTION: sample random tokens, decode, validate with RDKit, retry on failure/oversize.
def random_selfies_molecule(vocab, min_len=SELFIES_MIN_LEN, max_len=SELFIES_MAX_LEN, max_tries=20):
    for _ in range(max_tries):
        length = random.randint(min_len, max_len)
        tokens = [random.choice(vocab) for _ in range(length)]
        try:
            smiles = sf.decoder("".join(tokens))
        except Exception:
            continue
        mol = Chem.MolFromSmiles(smiles)
        if mol is None or mol.GetNumHeavyAtoms() > MAX_HEAVY_ATOMS:
            continue
        return Chem.MolToSmiles(mol)
    return None

for _ in range(5):
    print(random_selfies_molecule(VOCAB))


In [ ]:
def draw_molecule_grid(smiles_list, legends=None, mols_per_row=4, sub_img_size=(200, 160)):
    """Grid image of a list of SMILES -- reused below for the GA's output too."""
    legends = legends or [""] * len(smiles_list)
    pairs = [(Chem.MolFromSmiles(s), leg) for s, leg in zip(smiles_list, legends)]
    pairs = [(m, leg) for m, leg in pairs if m is not None]
    mols, legends = ([m for m, _ in pairs], [leg for _, leg in pairs]) if pairs else ([], [])
    return Draw.MolsToGridImage(mols, molsPerRow=mols_per_row, subImgSize=sub_img_size, legends=legends)


# What does "randomly generated, from this vocabulary" actually look like? These molecules are NOT
# evaluated by the oracle here -- purely structural, to see what's reachable before any search runs.
_space_preview = []
while len(_space_preview) < 16:
    smi = random_selfies_molecule(VOCAB)
    if smi is not None:
        _space_preview.append(smi)

_legends = [f"{Chem.MolFromSmiles(s).GetNumHeavyAtoms()} heavy atoms" for s in _space_preview]
draw_molecule_grid(_space_preview, legends=_legends)


In [ ]:
def run_progress_curve(evaluate_fn, candidate_smiles_list):
    """Evaluate each candidate once (in order) and track the best score seen so far.
    Returns (evaluated_smiles, scores, best_so_far) -- candidates the oracle couldn't score are
    skipped entirely (they never used a real oracle call's worth of information)."""
    evaluated, scores, best_so_far = [], [], []
    best = float("inf")
    for smi in tqdm(candidate_smiles_list, desc="Docking candidates"):
        score = evaluate_fn(smi)
        if score is None:
            continue
        evaluated.append(smi)
        scores.append(score)
        best = min(best, score)
        best_so_far.append(best)
    return evaluated, scores, best_so_far


def plot_progress(curves, title):
    plt.figure(figsize=(5.5, 3.4))
    for label, best_so_far in curves.items():
        plt.plot(range(1, len(best_so_far) + 1), best_so_far, marker=".", label=label)
    plt.xlabel("real oracle calls")
    plt.ylabel("best score so far (lower = better)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
N_RANDOM = 25

random_candidates = []
while len(random_candidates) < N_RANDOM:
    smi = random_selfies_molecule(VOCAB)
    if smi is not None:
        random_candidates.append(smi)

random_smiles, random_scores, random_best_so_far = run_progress_curve(evaluate, random_candidates)
print(f"Method 1 (random search): best score found = {min(random_scores):.3f}  "
      f"({len(random_smiles)} molecules evaluated)")
plot_progress({"random search": random_best_so_far}, "Method 1 -- random search")


In [ ]:
# Optional: evaluate a different number of random molecules interactively. The search SPACE
# (VOCAB/length range/heavy-atom cap) stays fixed -- only how many you sample changes.
import ipywidgets as widgets
from IPython.display import display, clear_output

n_random_slider = widgets.IntSlider(value=25, min=5, max=75, step=5, description="N_RANDOM")
run_random_button = widgets.Button(description="Run random search", button_style="success")
random_output = widgets.Output()


def on_random_run_clicked(_):
    with random_output:
        clear_output(wait=True)
        candidates = []
        while len(candidates) < n_random_slider.value:
            smi = random_selfies_molecule(VOCAB)
            if smi is not None:
                candidates.append(smi)
        smis, scores, curve = run_progress_curve(evaluate, candidates)
        plot_progress({"random search (demo)": curve}, f"best = {min(scores):.3f} kcal/mol")


run_random_button.on_click(on_random_run_clicked)
display(widgets.VBox([n_random_slider, run_random_button, random_output]))


## (c) · Method 2 — a genetic algorithm

A genetic algorithm (GA) evolves a *population* of molecules over several generations:

1. **Selection** — pick parents biased toward better-scoring individuals (here: tournament
   selection, the same style used in the classic graph-based molecular GAs, e.g. Jensen 2019).
2. **Crossover** — combine two parents' SELFIES tokens into a child.
3. **Mutation** — randomly swap a few tokens, so the population doesn't get stuck.
4. **Elitism** — always keep the best individual(s) unchanged, so a good molecule is never lost.

Unlike random search, a GA *uses* the oracle's feedback to bias where it looks next — the whole
point of this notebook is to see how much that buys you for the same number of oracle calls.


In [ ]:
def smiles_to_tokens(smiles):
    return list(sf.split_selfies(sf.encoder(smiles)))


def tokens_to_smiles(tokens, max_heavy_atoms=MAX_HEAVY_ATOMS):
    try:
        smiles = sf.decoder("".join(tokens))
    except Exception:
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumHeavyAtoms() > max_heavy_atoms:
        return None
    return Chem.MolToSmiles(mol)


def crossover(tokens_a, tokens_b):
    """Single-point crossover: swap the tails of two token lists to make one child."""
    if len(tokens_a) < 2 or len(tokens_b) < 2:
        return list(random.choice([tokens_a, tokens_b]))
    cut_a = random.randint(1, len(tokens_a) - 1)
    cut_b = random.randint(1, len(tokens_b) - 1)
    return tokens_a[:cut_a] + tokens_b[cut_b:]


In [ ]:
# SOLUTION: independently replace each token with probability p, returning a new list.
def mutate_selfies(tokens, vocab, p=0.15):
    new_tokens = []
    for tok in tokens:
        new_tokens.append(random.choice(vocab) if random.random() < p else tok)
    return new_tokens

print(mutate_selfies(smiles_to_tokens(DEMO_SMILES), VOCAB, p=0.3))


In [ ]:
# SOLUTION: sample k distinct indices, return the individual with the lowest score.
def tournament_select(population, scores, k=3):
    idxs = random.sample(range(len(population)), k=min(k, len(population)))
    best_idx = min(idxs, key=lambda i: scores[i])
    return population[best_idx]

_demo_pop = random.sample(SEED_SMILES, 3)
_demo_scores = [evaluate(s) or 0.0 for s in _demo_pop]
print("tournament winner:", tournament_select(_demo_pop, _demo_scores, k=3))


### GA hyperparameters

- **`GA_POP` × `GA_GENERATIONS`** — your total oracle-call budget for this method (population scored
  once initially, then once per generation).
- **`GA_TOURNAMENT_K`** — how many individuals compete for each parent slot. Small `k` (e.g. 2) keeps
  selection close to random (more exploration, slower convergence); large `k` almost always picks the
  current best (more exploitation, faster convergence, higher risk of getting stuck).
- **`GA_MUTATION_P`** — the per-token chance of a random swap. Too low and the population converges
  onto whatever it started near; too high and it's barely better than random search.
- **`GA_ELITISM`** — how many top individuals are copied over unchanged each generation. This is what
  guarantees the "best so far" curve below can only improve, never regress.

**The search space itself is unchanged from part (b)** — same `VOCAB`, same `SELFIES_MIN_LEN`/
`SELFIES_MAX_LEN`, same `MAX_HEAVY_ATOMS`. Only the *sampling strategy* is different, which is exactly
what makes comparing the two methods' oracle-call efficiency a fair comparison later on.


In [ ]:
GA_POP = 10
GA_GENERATIONS = 5
GA_MUTATION_P = 0.15
GA_TOURNAMENT_K = 3
GA_ELITISM = 1


def run_genetic_algorithm(evaluate_fn, vocab, pop_size=GA_POP, generations=GA_GENERATIONS,
                           mutation_p=GA_MUTATION_P, tournament_k=GA_TOURNAMENT_K, elitism=GA_ELITISM):
    # Initial population is drawn the same way random search's candidates are (random_selfies_molecule,
    # no seed-molecule warm start) -- otherwise the GA would get an unfair head start from real,
    # already-good scaffolds and the sample-efficiency comparison below wouldn't be apples-to-apples.
    population = []
    while len(population) < pop_size:
        smi = random_selfies_molecule(vocab)
        if smi is not None:
            population.append(smi)

    all_smiles, all_scores, best_so_far = [], [], []
    best = float("inf")
    pbar = tqdm(total=pop_size * (generations + 1), desc="GA oracle calls")

    def score_population(pop):
        nonlocal best
        pop_scores = []
        for smi in pop:
            s = evaluate_fn(smi)
            pbar.update(1)
            pop_scores.append(s if s is not None else float("inf"))
            if s is not None:
                all_smiles.append(smi)
                all_scores.append(s)
                best = min(best, s)
                best_so_far.append(best)
        return pop_scores

    scores = score_population(population)
    for gen in range(generations):
        ranked = sorted(zip(population, scores), key=lambda pair: pair[1])
        next_population = [smi for smi, _ in ranked[:elitism]]
        while len(next_population) < pop_size:
            parent_a = tournament_select(population, scores, k=tournament_k)
            parent_b = tournament_select(population, scores, k=tournament_k)
            child_tokens = crossover(smiles_to_tokens(parent_a), smiles_to_tokens(parent_b))
            child_tokens = mutate_selfies(child_tokens, vocab, p=mutation_p)
            child_smi = tokens_to_smiles(child_tokens)
            if child_smi is not None:
                next_population.append(child_smi)
        population = next_population
        scores = score_population(population)
        print(f"  generation {gen + 1}/{generations}  best so far = {best:.3f}")
    pbar.close()

    return all_smiles, all_scores, best_so_far


ga_smiles, ga_scores, ga_best_so_far = run_genetic_algorithm(evaluate, VOCAB)
print(f"Method 2 (genetic algorithm): best score found = {min(ga_scores):.3f}  "
      f"({len(ga_smiles)} molecules evaluated)")
plot_progress({"genetic algorithm": ga_best_so_far}, "Method 2 -- genetic algorithm")


In [ ]:
# What did the GA actually find? Show the top individuals it evaluated, best first.
_top_n = 8
_ranked_ga = sorted(zip(ga_smiles, ga_scores), key=lambda pair: pair[1])[:_top_n]
draw_molecule_grid([smi for smi, _ in _ranked_ga],
                    legends=[f"{score:.2f} kcal/mol" for _, score in _ranked_ga])


In [ ]:
# Optional: explore GA hyperparameters interactively. Try, for example, mutation_p=0.0 (no
# diversity injection) vs. 0.6 (too disruptive), or population_size=3 (too little diversity to
# start with). The progress bar and molecule grid update on every run.
mutation_slider = widgets.FloatSlider(value=0.15, min=0.0, max=0.6, step=0.05, description="mutation p")
pop_slider = widgets.IntSlider(value=10, min=4, max=16, step=2, description="population")
gen_slider = widgets.IntSlider(value=5, min=1, max=8, step=1, description="generations")
run_button = widgets.Button(description="Run GA", button_style="success")
ga_output = widgets.Output()


def on_ga_run_clicked(_):
    with ga_output:
        clear_output(wait=True)
        smis, scores, curve = run_genetic_algorithm(
            evaluate, VOCAB,
            pop_size=pop_slider.value, generations=gen_slider.value, mutation_p=mutation_slider.value,
        )
        plot_progress({"genetic algorithm (demo)": curve}, f"best = {min(scores):.3f} kcal/mol")
        _ranked_demo = sorted(zip(smis, scores), key=lambda pair: pair[1])[:8]
        display(draw_molecule_grid([smi for smi, _ in _ranked_demo],
                                    legends=[f"{score:.2f} kcal/mol" for _, score in _ranked_demo]))


run_button.on_click(on_ga_run_clicked)
display(widgets.VBox([mutation_slider, pop_slider, gen_slider, run_button, ga_output]))


## (d) · Method 3 — surrogate-guided active learning (Bayesian-optimization-style)

The GA above still calls the real oracle on every single individual it generates. If the oracle were
even more expensive (imagine a wet-lab assay instead of a 3-second docking run), that would be
wasteful. **Active learning** fixes this: train a cheap surrogate model to *predict* the oracle's
score, use it to cheaply screen a large pool of candidates, and only spend real oracle calls on the
handful the surrogate is most excited about.

We use a `RandomForestRegressor` on Morgan fingerprints as the surrogate. A random forest gives us
an uncertainty estimate for free: the *disagreement* between its individual trees. We rank candidates
by a UCB-style (Upper Confidence Bound) acquisition score, `mean - beta * std` (remember: lower is
better here, so we want candidates the surrogate predicts as good AND is uncertain about — that's
where the real oracle's feedback is most valuable).


In [ ]:
FP_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def featurize(smiles_list):
    fps = np.zeros((len(smiles_list), 1024), dtype=np.float32)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            arr = np.zeros(1024, dtype=np.float32)
            DataStructs.ConvertToNumpyArray(FP_GENERATOR.GetFingerprint(mol), arr)
            fps[i] = arr
    return fps


print(featurize(SEED_SMILES[:2]).shape)


### Pool size vs. feature space — two different numbers

Two things below are easy to conflate:

- **`AL_POOL_SIZE`** — how many *candidate molecules* are considered each round. The pool is built by
  mutating already-labeled molecules (same `VOCAB`/`MAX_HEAVY_ATOMS` constraints as everywhere else in
  this notebook) — it's "how many," not "in what representation."
- **The surrogate's feature space** — before `RandomForestRegressor` ever sees a pool molecule, it's
  mapped to a **1024-bit Morgan/ECFP fingerprint** (`featurize`, above). That fixed 1024-dimensional
  space is what the acquisition function actually scores candidates in, regardless of pool size.

Making the pool bigger only makes the *cheap* side of the loop — generating mutants, fingerprinting
them, running `rf.predict` across all trees — do more work; it does **not** change how many *real*
(expensive, docked) oracle calls happen per round. That's fixed at `AL_ACQUIRE_PER_ROUND`. The cell
right after the active-learning run below measures this trade-off directly.


In [ ]:
# The surrogate only ever sees a handful of labeled molecules (as few as AL_INITIAL=12) mapped into a
# 1024-dimensional fingerprint space -- deep, unconstrained trees would happily memorize that tiny
# training set instead of generalizing. Capping tree depth and requiring several samples per leaf
# keeps the forest closer to a smoothed nearest-neighbor model, which is what we actually want an
# acquisition function to rank candidates by. Used everywhere below instead of a bare
# RandomForestRegressor(n_estimators=..., random_state=...).
RF_HYPERPARAMS = dict(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=SEED,
)

In [ ]:
AL_INITIAL = 12
AL_ROUNDS = 5
AL_ACQUIRE_PER_ROUND = 3
AL_POOL_SIZE = 60
AL_BETA = 1.0

# SOLUTION: per-tree predictions -> ensemble mean/std -> UCB score (mean - beta*std) -> top_k lowest.
def rank_by_acquisition(rf, pool_smiles, beta=AL_BETA, top_k=AL_ACQUIRE_PER_ROUND):
    X_pool = featurize(pool_smiles)
    tree_preds = np.array([tree.predict(X_pool) for tree in rf.estimators_])
    mean = tree_preds.mean(axis=0)
    std = tree_preds.std(axis=0)
    acquisition = mean - beta * std
    order = np.argsort(acquisition)
    return [pool_smiles[i] for i in order[:top_k]]


In [ ]:
def run_active_learning(evaluate_fn, vocab, initial=AL_INITIAL, rounds=AL_ROUNDS,
                         acquire_per_round=AL_ACQUIRE_PER_ROUND, pool_size=AL_POOL_SIZE, beta=AL_BETA):
    labeled_smiles, labeled_scores, best_so_far = [], [], []
    best = float("inf")
    pbar = tqdm(total=initial + rounds * acquire_per_round, desc="Active-learning oracle calls")

    def label(smi):
        nonlocal best
        score = evaluate_fn(smi)
        pbar.update(1)
        if score is None:
            return
        labeled_smiles.append(smi)
        labeled_scores.append(score)
        best = min(best, score)
        best_so_far.append(best)

    while len(labeled_smiles) < initial:
        smi = random_selfies_molecule(vocab)
        if smi is not None:
            label(smi)

    for rnd in range(rounds):
        rf = RandomForestRegressor(**RF_HYPERPARAMS)
        rf.fit(featurize(labeled_smiles), labeled_scores)

        pool = []
        while len(pool) < pool_size:
            tokens = mutate_selfies(smiles_to_tokens(random.choice(labeled_smiles)), vocab, p=0.2)
            smi = tokens_to_smiles(tokens)
            if smi is not None:
                pool.append(smi)

        for smi in rank_by_acquisition(rf, pool, beta=beta, top_k=acquire_per_round):
            label(smi)
        print(f"  round {rnd + 1}/{rounds}  best so far = {best:.3f}")
    pbar.close()

    return labeled_smiles, labeled_scores, best_so_far


al_smiles, al_scores, al_best_so_far = run_active_learning(evaluate, VOCAB)
print(f"Method 3 (surrogate-guided active learning): best score found = {min(al_scores):.3f}  "
      f"({len(al_smiles)} molecules evaluated)")
plot_progress({"surrogate-guided": al_best_so_far}, "Method 3 -- surrogate-guided active learning")

### Surrogate diagnostics — is the random forest worth trusting, and what does it cost?

Active learning only pays off if two things are both true: the surrogate's predictions actually
generalize to molecules it wasn't trained on (otherwise the acquisition function ranks the pool almost
randomly), and training/using it is cheap next to the real oracle it's meant to save calls on. Both
checks below reuse the labeled data (`al_smiles`/`al_scores`) the active-learning run above already
collected — **no extra oracle/docking calls**.


In [ ]:
# Held-out fitness of the RF surrogate: 5-fold cross-validated R^2/MAE, using the SAME capped
# RF_HYPERPARAMS the active-learning loop itself trains with. Compared against in-sample
# (training-set) R^2 -- if that's much higher than the cross-validated number, the forest is
# overfitting rather than actually predicting.
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error

X_al = featurize(al_smiles)
y_al = np.array(al_scores)

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
y_pred_cv = cross_val_predict(RandomForestRegressor(**RF_HYPERPARAMS), X_al, y_al, cv=cv)

cv_r2 = r2_score(y_al, y_pred_cv)
cv_mae = mean_absolute_error(y_al, y_pred_cv)

rf_train_fit = RandomForestRegressor(**RF_HYPERPARAMS).fit(X_al, y_al)
train_r2 = r2_score(y_al, rf_train_fit.predict(X_al))

print(f"5-fold cross-validated R^2  : {cv_r2:6.3f}")
print(f"5-fold cross-validated MAE  : {cv_mae:6.3f} kcal/mol")
print(f"in-sample (training) R^2    : {train_r2:6.3f}   <- much higher than CV R^2 would mean overfitting")

plt.figure(figsize=(4.5, 4.5))
lims = [min(y_al.min(), y_pred_cv.min()), max(y_al.max(), y_pred_cv.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="perfect prediction")
plt.scatter(y_al, y_pred_cv, s=25)
plt.xlabel("true docking score (kcal/mol)")
plt.ylabel("cross-validated prediction (kcal/mol)")
plt.title("Random forest surrogate: held-out fit")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# How expensive is the RF surrogate itself, next to a real oracle call? Training + inference happen
# once per active-learning round; the oracle is called AL_ACQUIRE_PER_ROUND times per round. This
# times both sides on the wall clock, reusing ORACLE_TIMING captured near the top of the notebook so
# this cell makes ZERO additional real oracle/docking calls.
t0 = time.time()
_cost_rf = RandomForestRegressor(**RF_HYPERPARAMS)
_cost_rf.fit(featurize(al_smiles), al_scores)
rf_train_time = time.time() - t0

_cost_pool = []
while len(_cost_pool) < AL_POOL_SIZE:
    tokens = mutate_selfies(smiles_to_tokens(random.choice(al_smiles)), VOCAB, p=0.2)
    smi = tokens_to_smiles(tokens)
    if smi is not None:
        _cost_pool.append(smi)

t0 = time.time()
rank_by_acquisition(_cost_rf, _cost_pool, beta=AL_BETA, top_k=AL_ACQUIRE_PER_ROUND)
rf_inference_time = time.time() - t0

rf_round_time = rf_train_time + rf_inference_time
oracle_call_time = ORACLE_TIMING[EXHAUSTIVENESS_CHEAP]
oracle_round_time = oracle_call_time * AL_ACQUIRE_PER_ROUND

print(f"RF train time                       : {rf_train_time * 1000:7.1f} ms   ({len(al_smiles)} labeled molecules)")
print(f"RF inference time (pool={AL_POOL_SIZE:3d})        : {rf_inference_time * 1000:7.1f} ms")
print(f"RF surrogate cost / round            : {rf_round_time * 1000:7.1f} ms")
print(f"One oracle (docking) call             : {oracle_call_time:7.2f} s")
print(f"{AL_ACQUIRE_PER_ROUND} oracle calls / round (real work) : {oracle_round_time:7.2f} s")
print(f"\nThe surrogate adds about {100 * rf_round_time / oracle_round_time:.2f}% of one round's real "
      f"oracle time -- effectively free next to docking, which is exactly why it's worth training one "
      f"even though it's an imperfect model (see the fitness check above).")

In [ ]:
# How does compute scale with pool size? This reuses the labeled data the run above already
# collected, purely to measure the CHEAP side of the loop (generating mutants, fingerprinting them,
# then rf.predict across every tree) -- it makes ZERO real oracle/docking calls, on purpose, so you
# can see this cost is decoupled from AL_ACQUIRE_PER_ROUND (which is what actually costs real time).
_bench_rf = RandomForestRegressor(**RF_HYPERPARAMS)
_bench_rf.fit(featurize(al_smiles), al_scores)

for pool_size_try in [20, 60, 120, 240]:
    pool = []
    while len(pool) < pool_size_try:
        tokens = mutate_selfies(smiles_to_tokens(random.choice(al_smiles)), VOCAB, p=0.2)
        smi = tokens_to_smiles(tokens)
        if smi is not None:
            pool.append(smi)
    t0 = time.time()
    rank_by_acquisition(_bench_rf, pool, beta=AL_BETA, top_k=AL_ACQUIRE_PER_ROUND)
    dt = time.time() - t0
    print(f"pool_size={pool_size_try:4d}   time={dt * 1000:6.1f} ms   (0 real oracle/docking calls)")

## (e) · Method 4 — VAE latent-space active learning

Method 3's surrogate ranks candidates built by mutating SELFIES *tokens* directly. Another common
approach: learn a smooth, continuous embedding of the same molecule space with a **variational
autoencoder (VAE)**, then search *inside that latent space* instead of at token-edit distance. Nearby
latent points tend to decode to structurally related molecules, so a small Gaussian step in latent
space is a different (often gentler) way to propose a neighbor than randomly swapping SELFIES tokens.

We load a small **pretrained** VAE (~560 KB, GRU encoder/decoder, PyTorch) instead of training one
live — training even a tiny sequence VAE well enough to be useful takes longer than a class-sized
budget affords. It was trained *offline*, on the exact same space every other method here draws from:
`VOCAB`, `random_selfies_molecule`'s uniform token sampling, the same `SELFIES_MIN_LEN`/`MAX_LEN` and
`MAX_HEAVY_ATOMS` — no seed-molecule warm start, for the same fairness reason the GA's initial
population isn't warm-started from `SEED_SMILES` either.

The active-learning loop itself is otherwise identical to Method 3: the same `RF_HYPERPARAMS` random
forest (fast to train, gives predictions *and* uncertainty for free via tree disagreement), the same
UCB acquisition, the same oracle-call budget (`AL_INITIAL`/`AL_ROUNDS`/`AL_ACQUIRE_PER_ROUND`). The
only thing that changes is *where* candidate neighbors come from: Gaussian perturbation of a labeled
molecule's latent vector, decoded back to SMILES, instead of mutating its SELFIES tokens.


In [ ]:
!pip -q install torch > /dev/null
import torch
import torch.nn as nn

# Committed directly in this tutorial's own repo (tutorials/generative-vae/vae_checkpoint.pt) --
# small enough (~560 KB) to fetch the same way protein.pdb was fetched in part (a), no external
# hosting/account needed.
VAE_CHECKPOINT_URL = (
    "https://raw.githubusercontent.com/julschleinitz/ai4chemistry-bootcamp/main/"
    "tutorials/generative-vae/vae_checkpoint.pt"
)
urllib.request.urlretrieve(VAE_CHECKPOINT_URL, "vae_checkpoint.pt")
VAE_CHECKPOINT = torch.load("vae_checkpoint.pt", weights_only=False)

print("checkpoint trained on", VAE_CHECKPOINT["trained_on"]["n_corpus"], "molecules  |  "
      "latent_dim =", VAE_CHECKPOINT["hyperparams"]["latent_dim"])

In [ ]:
class LatentEncoder(nn.Module):
    """GRU encoder: token sequence -> (mu, logvar) of the latent Gaussian."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.to_mu = nn.Linear(hidden_dim, latent_dim)
        self.to_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        _, h = self.gru(self.embed(x))
        h = h.squeeze(0)
        return self.to_mu(h), self.to_logvar(h)


class LatentDecoder(nn.Module):
    """GRU decoder: latent vector -> token sequence, generated autoregressively (greedy)."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, pad_idx, sos_idx):
        super().__init__()
        self.sos_idx = sos_idx
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.from_z = nn.Linear(latent_dim, hidden_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def generate(self, z, max_len):
        h = self.from_z(z).unsqueeze(0)
        tok = torch.full((z.size(0),), self.sos_idx, dtype=torch.long)
        outputs = []
        for _ in range(max_len):
            emb = self.embed(tok).unsqueeze(1)
            out, h = self.gru(emb, h)
            tok = self.out(out.squeeze(1)).argmax(dim=-1)
            outputs.append(tok)
        return torch.stack(outputs, dim=1)


_hp = VAE_CHECKPOINT["hyperparams"]
IDX2TOK = VAE_CHECKPOINT["idx2tok"]
TOK2IDX = {t: i for i, t in enumerate(IDX2TOK)}
PAD_TOK, SOS_TOK, EOS_TOK, UNK_TOK = "<pad>", "<sos>", "<eos>", "<unk>"
LATENT_DIM, VAE_MAX_LEN = _hp["latent_dim"], _hp["max_len"]

vae_encoder = LatentEncoder(len(IDX2TOK), _hp["embed_dim"], _hp["hidden_dim"], LATENT_DIM, TOK2IDX[PAD_TOK])
vae_decoder = LatentDecoder(len(IDX2TOK), _hp["embed_dim"], _hp["hidden_dim"], LATENT_DIM,
                             TOK2IDX[PAD_TOK], TOK2IDX[SOS_TOK])
vae_encoder.load_state_dict(VAE_CHECKPOINT["encoder_state_dict"])
vae_decoder.load_state_dict(VAE_CHECKPOINT["decoder_state_dict"])
vae_encoder.eval()
vae_decoder.eval()

print(f"VAE ready: {len(IDX2TOK)} tokens (incl. special tokens), latent_dim = {LATENT_DIM}")

In [ ]:
def encode_smiles_to_latent(smiles_list):
    """SMILES -> mean latent vector (mu, not a sampled z) -- deterministic, used both to embed the
    labeled set for the RF surrogate and as the anchor for latent-space perturbation below."""
    rows = []
    for smi in smiles_list:
        ids = [TOK2IDX.get(t, TOK2IDX[UNK_TOK]) for t in smiles_to_tokens(smi)]
        ids = ids[:VAE_MAX_LEN - 1] + [TOK2IDX[EOS_TOK]]
        ids += [TOK2IDX[PAD_TOK]] * (VAE_MAX_LEN - len(ids))
        rows.append(ids)
    with torch.no_grad():
        mu, _ = vae_encoder(torch.tensor(rows, dtype=torch.long))
    return mu.numpy()


def decode_latent_to_smiles(z_batch):
    """Latent vectors -> SMILES, reusing tokens_to_smiles for the same validity/heavy-atom check
    every other method uses -- None wherever decoding fails or the heavy-atom cap is exceeded."""
    with torch.no_grad():
        gen_ids = vae_decoder.generate(torch.tensor(z_batch, dtype=torch.float32), VAE_MAX_LEN)
    decoded = []
    for row in gen_ids.tolist():
        toks = []
        for idx in row:
            if idx == TOK2IDX[EOS_TOK]:
                break
            if idx in (TOK2IDX[PAD_TOK], TOK2IDX[UNK_TOK]):
                continue
            toks.append(IDX2TOK[idx])
        decoded.append(tokens_to_smiles(toks))
    return decoded


# Sanity check: encode the seed molecules, decode them straight back -- most should round-trip
# to the exact same canonical SMILES (this is a lossy autoencoder trained on a tiny vocabulary,
# not a hash function, so an occasional near-miss is expected).
_demo_latents = encode_smiles_to_latent(SEED_SMILES)
for original, reconstructed in zip(SEED_SMILES, decode_latent_to_smiles(_demo_latents)):
    print(f"{original:32s} -> {reconstructed}")

In [ ]:
VAE_LATENT_SIGMA = 0.5  # stddev of the Gaussian step taken in latent space to propose a neighbor


def rank_latents_by_acquisition(rf, pool_latents, beta=AL_BETA, top_k=AL_ACQUIRE_PER_ROUND):
    """Same UCB math as rank_by_acquisition (Method 3) -- mean - beta*std across the RF's trees --
    just applied directly to latent vectors instead of Morgan fingerprints."""
    tree_preds = np.array([tree.predict(pool_latents) for tree in rf.estimators_])
    mean = tree_preds.mean(axis=0)
    std = tree_preds.std(axis=0)
    acquisition = mean - beta * std
    order = np.argsort(acquisition)
    return order[:top_k]

In [ ]:
def run_vae_active_learning(evaluate_fn, initial=AL_INITIAL, rounds=AL_ROUNDS,
                             acquire_per_round=AL_ACQUIRE_PER_ROUND, pool_size=AL_POOL_SIZE,
                             beta=AL_BETA, sigma=VAE_LATENT_SIGMA):
    labeled_smiles, labeled_scores, labeled_latents, best_so_far = [], [], [], []
    best = float("inf")
    pbar = tqdm(total=initial + rounds * acquire_per_round, desc="VAE active-learning oracle calls")

    def label(smi):
        nonlocal best
        score = evaluate_fn(smi)
        pbar.update(1)
        if score is None:
            return
        labeled_smiles.append(smi)
        labeled_scores.append(score)
        labeled_latents.append(encode_smiles_to_latent([smi])[0])
        best = min(best, score)
        best_so_far.append(best)

    while len(labeled_smiles) < initial:
        smi = random_selfies_molecule(VOCAB)
        if smi is not None:
            label(smi)

    for rnd in range(rounds):
        rf = RandomForestRegressor(**RF_HYPERPARAMS)
        rf.fit(np.array(labeled_latents), labeled_scores)

        # Pool candidates come from a Gaussian step around an already-labeled molecule's latent
        # vector, decoded back to SMILES -- the latent-space analogue of Method 3's token mutation.
        pool_latents, pool_smiles = [], []
        while len(pool_smiles) < pool_size:
            base_z = labeled_latents[random.randrange(len(labeled_latents))]
            z = base_z + sigma * np.random.randn(LATENT_DIM)
            smi = decode_latent_to_smiles(z[None, :])[0]
            if smi is not None:
                pool_latents.append(z)
                pool_smiles.append(smi)
        pool_latents = np.array(pool_latents)

        for i in rank_latents_by_acquisition(rf, pool_latents, beta=beta, top_k=acquire_per_round):
            label(pool_smiles[i])
        print(f"  round {rnd + 1}/{rounds}  best so far = {best:.3f}")
    pbar.close()

    return labeled_smiles, labeled_scores, best_so_far


vae_al_smiles, vae_al_scores, vae_al_best_so_far = run_vae_active_learning(evaluate)
print(f"Method 4 (VAE latent-space active learning): best score found = {min(vae_al_scores):.3f}  "
      f"({len(vae_al_smiles)} molecules evaluated)")
plot_progress({"VAE latent-space": vae_al_best_so_far}, "Method 4 -- VAE latent-space active learning")

## Comparison — sample efficiency

All four methods used a comparable, small number of *real* oracle calls. Which one found the best
molecule fastest?

In [ ]:
plot_progress({
    "random search": random_best_so_far,
    "genetic algorithm": ga_best_so_far,
    "surrogate-guided": al_best_so_far,
    "VAE latent-space": vae_al_best_so_far,
}, "Sample efficiency: best score so far vs. real oracle calls")

print(f"random search        best = {min(random_scores):.3f} kcal/mol")
print(f"genetic algorithm    best = {min(ga_scores):.3f} kcal/mol")
print(f"surrogate-guided     best = {min(al_scores):.3f} kcal/mol")
print(f"VAE latent-space     best = {min(vae_al_scores):.3f} kcal/mol")

**Questions to discuss:**

1. Which method found its best molecule using the fewest oracle calls? Does that match your
   intuition about how each method uses feedback from previous evaluations?
2. If your real oracle were 100x more expensive (say, 30 total calls instead of ~25-40 per method),
   which method would you trust most, and why?
3. Docking scores correlate only loosely with true binding affinity (T015 makes this point too).
   What would you want to check before trusting any of these "best" molecules?
4. All four methods here only ever generate small, drug-like-ish molecules because of `MAX_HEAVY_ATOMS`
   and the seed vocabulary. How would removing that constraint change what "winning" looks like?
5. Methods 3 and 4 share the same surrogate/acquisition machinery and only differ in how they propose
   candidates (token mutation vs. latent-space perturbation). Did that change the molecules found, the
   sample efficiency, both, or neither -- and what does that tell you about how much the *representation*
   a search operates in matters versus the *search strategy* itself?

## Challenge · Submit your best candidate to the leaderboard

Pick a **method**, its **parameters**, and an **oracle budget** (total real docking calls it's
allowed), then run the cell below. It re-runs that one method with your settings, shows the best
molecule it found, and submits it to the leaderboard together with exactly what you ran -- so you
can try several configurations and see which one actually earns its budget.

**Unlike the fine-tuning tutorial, you are not submitting your own score.** Your cheap-settings
docking numbers (`exhaustiveness=1, num_modes=1`) are for guiding your own search during class. The
leaderboard is graded independently, for everyone, using the docking tutorial's actual demo settings
(`exhaustiveness=8, num_modes=10`) — so every team's score is directly comparable, and nobody can get
a lucky number from an under-powered docking run. You can submit as many times as you like; the
leaderboard keeps each team's best graded score. Your entry appears on the tutorial page's live
leaderboard within a few minutes of submitting.

In [ ]:
!pip -q install requests > /dev/null
import requests, uuid
from datetime import datetime, timezone

LEADERBOARD_ENDPOINT_URL = "https://script.google.com/macros/s/AKfycbyJQ0x9g_e6XUAK2i6LYArJHU8azphCqfiZhTvxX4Xp1A8lmIG36LUHg7fy2u2Nf_zn/exec"
MAX_SUBMIT_HEAVY_ATOMS = 50

team_name_box = widgets.Text(value="Solutions", description="Team name",
                              style={"description_width": "initial"}, layout=widgets.Layout(width="400px"))
method_dropdown = widgets.Dropdown(
    options=[
        ("Method 1 -- random search", "random"),
        ("Method 2 -- genetic algorithm", "genetic_algorithm"),
        ("Method 3 -- surrogate active learning (RF + token mutation)", "active_learning"),
        ("Method 4 -- VAE latent-space active learning", "vae_latent_active_learning"),
    ],
    value="genetic_algorithm", description="Method", style={"description_width": "initial"},
)
budget_slider = widgets.IntSlider(value=30, min=10, max=80, step=5, description="Oracle budget",
                                   style={"description_width": "initial"})

# Method-specific "shape" knobs -- these change HOW a method spends its calls, not how MANY it
# spends. The oracle budget above is the one knob every method shares, so configurations are
# comparable across methods at a matched real-docking-call cost.
ga_pop_box = widgets.IntSlider(value=10, min=4, max=16, step=2, description="population size")
mutation_p_box = widgets.FloatSlider(value=0.15, min=0.0, max=0.6, step=0.05, description="mutation p")
tournament_k_box = widgets.IntSlider(value=3, min=2, max=5, description="tournament k")
elitism_box = widgets.IntSlider(value=1, min=0, max=3, description="elitism")

pool_size_box = widgets.IntSlider(value=60, min=20, max=150, step=10, description="pool size")
beta_box = widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.25, description="UCB beta")
acquire_box = widgets.IntSlider(value=3, min=1, max=6, description="acquire/round")

sigma_box = widgets.FloatSlider(value=0.5, min=0.1, max=1.5, step=0.1, description="latent sigma")

param_panels = {
    "random": widgets.VBox([widgets.HTML(
        "<i>No extra parameters -- the oracle budget alone sets how many random molecules are drawn.</i>")]),
    "genetic_algorithm": widgets.VBox([ga_pop_box, mutation_p_box, tournament_k_box, elitism_box]),
    "active_learning": widgets.VBox([pool_size_box, beta_box, acquire_box]),
    "vae_latent_active_learning": widgets.VBox([pool_size_box, beta_box, acquire_box, sigma_box]),
}
param_box = widgets.VBox([param_panels[method_dropdown.value]])


def _on_method_change(change):
    param_box.children = [param_panels[change["new"]]]


method_dropdown.observe(_on_method_change, names="value")

run_button = widgets.Button(description="Run + submit", button_style="success")
challenge_output = widgets.Output()


def _split_budget(budget, acquire_per_round):
    """One total-oracle-call budget -> (initial, rounds) for an active-learning loop: roughly half
    the budget goes to the initial random batch, the rest is split across acquisition rounds."""
    initial = max(acquire_per_round + 1, budget // 2)
    rounds = max(1, (budget - initial) // acquire_per_round)
    return initial, rounds


def _format_params(params):
    return ", ".join(f"{k}={v}" for k, v in params.items())


def _run_challenge(_):
    with challenge_output:
        clear_output(wait=True)
        method = method_dropdown.value
        budget = budget_slider.value

        if method == "random":
            candidates = []
            while len(candidates) < budget:
                smi = random_selfies_molecule(VOCAB)
                if smi is not None:
                    candidates.append(smi)
            smis, scores, curve = run_progress_curve(evaluate, candidates)
            params_used = {"budget": budget}

        elif method == "genetic_algorithm":
            # A GA needs to score a full population at least twice (initial + >=1 generation) to do
            # anything at all, so clamp population size to at most half the budget -- otherwise even
            # one generation would blow through it.
            pop_size = max(2, min(ga_pop_box.value, budget // 2))
            generations = max(1, round(budget / pop_size) - 1)
            smis, scores, curve = run_genetic_algorithm(
                evaluate, VOCAB, pop_size=pop_size, generations=generations,
                mutation_p=mutation_p_box.value, tournament_k=tournament_k_box.value,
                elitism=elitism_box.value,
            )
            params_used = {"population_size": pop_size, "generations": generations,
                            "mutation_p": mutation_p_box.value, "tournament_k": tournament_k_box.value,
                            "elitism": elitism_box.value}

        elif method == "active_learning":
            acquire = acquire_box.value
            initial, rounds = _split_budget(budget, acquire)
            smis, scores, curve = run_active_learning(
                evaluate, VOCAB, initial=initial, rounds=rounds, acquire_per_round=acquire,
                pool_size=pool_size_box.value, beta=beta_box.value,
            )
            params_used = {"initial": initial, "rounds": rounds, "acquire_per_round": acquire,
                            "pool_size": pool_size_box.value, "beta": beta_box.value}

        else:  # vae_latent_active_learning
            acquire = acquire_box.value
            initial, rounds = _split_budget(budget, acquire)
            smis, scores, curve = run_vae_active_learning(
                evaluate, initial=initial, rounds=rounds, acquire_per_round=acquire,
                pool_size=pool_size_box.value, beta=beta_box.value, sigma=sigma_box.value,
            )
            params_used = {"initial": initial, "rounds": rounds, "acquire_per_round": acquire,
                            "pool_size": pool_size_box.value, "beta": beta_box.value,
                            "sigma": sigma_box.value}

        if not scores:
            print("No molecule was scored -- try a larger oracle budget.")
            return

        best_idx = int(np.argmin(scores))
        best_smiles, best_score = smis[best_idx], scores[best_idx]
        best_mol = Chem.MolFromSmiles(best_smiles)
        if best_mol is None or best_mol.GetNumHeavyAtoms() > MAX_SUBMIT_HEAVY_ATOMS:
            print(f"Best candidate {best_smiles} failed the submission check (invalid or over "
                  f"{MAX_SUBMIT_HEAVY_ATOMS} heavy atoms) -- try again.")
            return

        plot_progress({method: curve}, f"{method}: best = {best_score:.3f} kcal/mol")
        print(f"method: {method}")
        print(f"requested oracle budget: {budget}  |  actual oracle calls used: {len(smis)}")
        print(f"parameters: {_format_params(params_used)}")
        print(f"Best candidate: {best_smiles}  |  cheap-settings score: {best_score:.3f} kcal/mol")

        notes = (f"method={method} | requested_budget={budget} | oracle_calls_used={len(smis)} | "
                 f"{_format_params(params_used)} | cheap-settings score {best_score:.3f} kcal/mol")
        payload = {
            "action": "submit",
            "run_id": str(uuid.uuid4()),
            "timestamp_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
            "team_name": team_name_box.value,
            "smiles": best_smiles,
            "method_family": method,
            "notes": notes,
        }
        resp = requests.post(LEADERBOARD_ENDPOINT_URL, json=payload, timeout=15)
        print(f"\n{resp.status_code} {resp.text}")
        print("Queued for grading -- check the tutorial page's live leaderboard in a few minutes.")


run_button.on_click(_run_challenge)
display(widgets.VBox([team_name_box, method_dropdown, budget_slider, param_box, run_button, challenge_output]))

## Bonus · Multi-objective generation

Optimizing the docking score alone is a trap: nothing above stops these methods from finding a
molecule that docks beautifully but is a terrible drug (or a known toxicity liability). Two more
objectives — both cheap, both computed with RDKit alone, no extra oracle calls needed:

- **QED** (`Descriptors.qed`) — a composite drug-likeness score, higher is better.
- **Structural alerts** — RDKit's built-in PAINS and BRENK substructure-filter catalogs flag known
  problematic chemical patterns (reactive groups, assay interference, etc.); we count matches, lower
  (fewer) is better.

These are exactly the same two properties that the live leaderboard also scores you on, so you can
see below where your own queued submission is likely to land on these axes.


In [ ]:
def build_alert_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)


ALERT_CATALOG = build_alert_catalog()


def qed_and_toxicity_alerts(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    return Descriptors.qed(mol), len(ALERT_CATALOG.GetMatches(mol))


all_smiles_combined = random_smiles + ga_smiles + al_smiles + vae_al_smiles
all_scores_combined = random_scores + ga_scores + al_scores + vae_al_scores

qed_values, alert_counts = [], []
for smi in all_smiles_combined:
    q, a = qed_and_toxicity_alerts(smi)
    qed_values.append(q)
    alert_counts.append(a)

plt.figure(figsize=(5.5, 4))
sc = plt.scatter(all_scores_combined, qed_values, c=alert_counts, cmap="coolwarm", s=30)
plt.xlabel("docking score, kcal/mol (lower = better)")
plt.ylabel("QED (higher = more drug-like)")
plt.colorbar(sc, label="structural alerts (PAINS + BRENK)")
plt.title("Docking score vs. drug-likeness vs. structural alerts")
plt.tight_layout()
plt.show()

In [ ]:
# SOLUTION: a point is Pareto-efficient unless some other point dominates it (at least as good in
# every objective, strictly better in at least one).
def is_pareto_efficient(scores):
    scores = np.asarray(scores)
    n = len(scores)
    efficient = np.ones(n, dtype=bool)
    for i in range(n):
        if not efficient[i]:
            continue
        dominates = np.all(scores <= scores[i], axis=1) & np.any(scores < scores[i], axis=1)
        if np.any(dominates):
            efficient[i] = False
    return efficient

objective_matrix = np.column_stack([all_scores_combined, [-q for q in qed_values], alert_counts])
pareto_mask = is_pareto_efficient(objective_matrix)
print(f"{pareto_mask.sum()} / {len(pareto_mask)} candidates are Pareto-efficient across "
      f"(docking score, QED, structural alerts)")

scores_arr, qed_arr = np.array(all_scores_combined), np.array(qed_values)
plt.figure(figsize=(5.5, 4))
plt.scatter(scores_arr[~pareto_mask], qed_arr[~pareto_mask], color="gray", alpha=0.4, label="dominated")
plt.scatter(scores_arr[pareto_mask], qed_arr[pareto_mask], color="crimson", label="Pareto-efficient")
plt.xlabel("docking score, kcal/mol (lower = better)")
plt.ylabel("QED (higher = more drug-like)")
plt.title("Pareto front: docking score vs. drug-likeness")
plt.legend()
plt.tight_layout()
plt.show()


**Discussion.** A single scalar "docking score" hides real trade-offs. Two common ways to handle
multiple objectives:

- **Weighted-sum scalarization** — fold everything into one number (e.g. `evaluate()` in part (a)'s
  `descriptor_fallback_score` already does this, informally). Simple, but the weights encode a value
  judgement that's easy to get wrong, and it can miss good trade-off solutions entirely.
- **Pareto search** — keep the whole non-dominated front and let a human pick, based on context
  (target product profile, safety margins, synthesis cost) the objectives alone can't capture.

None of the methods in this notebook were built with QED/toxicity in the loop — as an extension,
try swapping `evaluate()`'s return value for a multi-objective scalarization (e.g.
`affinity - 5 * qed + 2 * toxicity_alerts`) and re-running the genetic algorithm from part (c). Does
the population's docking score get worse? Does its QED get better?


## Wrap-up and going further

You built and compared four genuinely different strategies for searching chemical space against a
real (if cheaply-configured) docking oracle: undirected random search, a population-based genetic
algorithm, and two surrogate-guided active-learning loops that spend real oracle calls only where a
cheap model is most uncertain -- one proposing neighbors by mutating SELFIES tokens, the other by
perturbing a pretrained VAE's continuous latent space. That shared idea — train a cheap surrogate,
use it to decide what's worth measuring next — is the same principle behind Bayesian optimization
more generally (see this bootcamp's Bayesian optimization tutorial), and it becomes more valuable,
not less, as the real oracle gets more expensive than a few-second docking call.

**To go further:**

- Real docking practice uses much higher `exhaustiveness`, multiple docking poses, ensemble docking
  across several protein conformations, and follow-up MD refinement — a docking score alone is a
  screening filter, not a final answer.
- The VAE here was deliberately tiny (trained offline on ~12k synthetic molecules from this
  notebook's own toy vocabulary) purely to demonstrate latent-space active learning inside a class
  budget. A real VAE-based design pipeline (e.g. Gómez-Bombarelli et al., below) trains on millions
  of real molecules and can take GPU-hours to learn a chemically meaningful latent space.
- T015 talktorial (the docking tutorial this notebook cheaply adapts):
  https://projects.volkamerlab.org/teachopencadd/talktorials/T015_protein_ligand_docking.html
- smina (the docking engine used throughout): https://sourceforge.net/projects/smina/
- Jensen, J. H. *A graph-based genetic algorithm and generative model/Monte Carlo tree search for the
  exploration of chemical space.* Chem. Sci. **2019**, 10, 3567–3572.
  https://doi.org/10.1039/C8SC05372C
- Nigam, A.; Pollice, R.; Krenn, M.; Gomes, G. dos P.; Aspuru-Guzik, A. *Beyond generative models: superfast
  traversal, optimization, novelty, exploration and discovery (STONED) algorithm for molecules using
  SELFIES.* Chem. Sci. **2021**, 12, 7079–7090. https://doi.org/10.1039/D1SC00231G
- Gómez-Bombarelli, R. et al. *Automatic Chemical Design Using a Data-Driven Continuous
  Representation of Molecules.* ACS Cent. Sci. **2018**, 4, 268–276.
  https://doi.org/10.1021/acscentsci.7b00572